# Molecular Dynamics Simulations

In [ ]:
import hoomd
import matplotlib
import numpy
import itertools
import math
import gsd.hoomd

%matplotlib inline
matplotlib.style.use("ggplot")
import matplotlib_inline
import matplotlib.pyplot as plt

matplotlib_inline.backend_inline.set_matplotlib_formats("svg")


import importlib
import viz_helpers
importlib.reload(viz_helpers)

from viz_helpers import render

In [ ]:
# Defining the Physics
dt = 0.005
kT = 1.5

r_cut_LJ = 2.5
buffer_LJ = 0.4
lj_mode = "none"   # choose: "none", "shift", or "xplor"

integrator = hoomd.md.Integrator(dt=dt)

cell = hoomd.md.nlist.Cell(buffer=buffer_LJ)

lj = hoomd.md.pair.LJ(nlist=cell, mode=lj_mode)
lj.params[("A", "A")] = dict(epsilon=1, sigma=1)
lj.r_cut[("A", "A")] = r_cut_LJ

if lj_mode == "xplor":
    lj.r_on[("A", "A")] = 2.0   # must be < r_cut_LJ

integrator.forces.append(lj)

nvt = hoomd.md.methods.ConstantVolume(
    filter=hoomd.filter.All(),
    thermostat=hoomd.md.methods.thermostats.Bussi(kT=kT))
integrator.methods.append(nvt)

In [ ]:
print(lj.mode)

## load the lattice and attach physics

In [ ]:
cpu = hoomd.device.CPU()
simulation = hoomd.Simulation(device=cpu, seed=1)
simulation.create_state_from_gsd(filename="lattice.gsd")
simulation.operations.integrator = integrator

In [ ]:
#To visualize
render(simulation.state.get_snapshot())

In [ ]:
#See that all particles are currently starting with zero velocity
snapshot = simulation.state.get_snapshot()
snapshot.particles.velocity[0:5]

In [ ]:
#Assigning random non-zero initial velocities
simulation.state.thermalize_particle_momenta(filter=hoomd.filter.All(), kT=1.5)

snapshot = simulation.state.get_snapshot()
snapshot.particles.velocity[0:5]

In [ ]:
# Attach thermodynamics
thermo = hoomd.md.compute.ThermodynamicQuantities(filter=hoomd.filter.All())
simulation.operations.computes.append(thermo)

# Initialize
simulation.run(0)

# Run a bit
simulation.run(1000)

# Check values
print("Temperature:", thermo.kinetic_temperature)
print("Pressure:", thermo.pressure)
print("Potential Energy:", thermo.potential_energy)

# Visual check
render(simulation.state.get_snapshot())

In [ ]:
1 / 2 * 1.5 * thermo.degrees_of_freedom

In [ ]:
thermo.kinetic_energy

In [ ]:
thermo.kinetic_temperature

In [ ]:
simulation.run(10000)

In [ ]:
render(simulation.state.get_snapshot())

In [ ]:
import numpy as np

def voxel_counts(snapshot, n_bins):
    # Positions (N,3)
    pos = np.array(snapshot.particles.position)

    # Box length (assuming cubic box)
    L = snapshot.configuration.box[0]

    # Shift from [-L/2, L/2] → [0, L]
    pos_shifted = pos + L / 2

    # Compute voxel indices
    bin_size = L / n_bins
    indices = np.floor(pos_shifted / bin_size).astype(int)

    # Handle edge case where pos == L
    indices = np.clip(indices, 0, n_bins - 1)

    # Create grid
    counts = np.zeros((n_bins, n_bins, n_bins), dtype=int)

    # Count particles
    for i, j, k in indices:
        counts[i, j, k] += 1

    # Return flattened list
    return counts.flatten().tolist()

In [ ]:
snap = simulation.state.get_snapshot()

counts = voxel_counts(snap, n_bins=4)

print(len(counts))
print(counts[:10])   # first few voxel counts

plt.figure(figsize=(6, 4))
plt.hist(counts, bins=20)
plt.xlabel("Particles per voxel")
plt.ylabel("Number of voxels")
plt.title("Voxel Occupancy Histogram")
plt.grid(alpha=0.3)
plt.show()

In [ ]:
thermo.kinetic_energy

In [ ]:
hoomd.write.GSD.write(state=simulation.state, filename="random.gsd", mode="wb")

## Compressing The System

In [ ]:
import math

import hoomd
import matplotlib

%matplotlib inline
matplotlib.style.use("ggplot")
import matplotlib_inline

matplotlib_inline.backend_inline.set_matplotlib_formats("svg")

In [ ]:
cpu = hoomd.device.CPU()
simulation = hoomd.Simulation(device=cpu, seed=1)
simulation.create_state_from_gsd(filename="random.gsd")

In [ ]:
integrator = hoomd.md.Integrator(dt=0.005)
cell = hoomd.md.nlist.Cell(buffer=0.4)
lj = hoomd.md.pair.LJ(nlist=cell)
lj.params[("A", "A")] = dict(epsilon=1, sigma=1)
lj.r_cut[("A", "A")] = 2.5
integrator.forces.append(lj)
nvt = hoomd.md.methods.ConstantVolume(
    filter=hoomd.filter.All(), thermostat=hoomd.md.methods.thermostats.Bussi(kT=1.5)
)
integrator.methods.append(nvt)
simulation.operations.integrator = integrator

In [ ]:
ramp = hoomd.variant.Ramp(A=0, B=1, t_start=simulation.timestep, t_ramp=20_000)

In [ ]:
steps = range(0, 40000, 20)
y = [ramp(step) for step in steps]

fig = matplotlib.figure.Figure(figsize=(5, 3.09))
ax = fig.add_subplot()
ax.plot(steps, y)
ax.set_xlabel("timestep")
ax.set_ylabel("ramp")
ax.tick_params("x", labelrotation=30)
fig

In [ ]:
rho = simulation.state.N_particles / simulation.state.box.volume
rho

In [ ]:
final_rho = 1.2
final_volume = simulation.state.N_particles / final_rho

In [ ]:
inverse_volume_ramp = hoomd.variant.box.InverseVolumeRamp(
    initial_box=simulation.state.box,
    final_volume=final_volume,
    t_start=simulation.timestep,
    t_ramp=20_000,
)

In [ ]:
steps = range(0, 40000, 20)
y = [inverse_volume_ramp(step)[0] for step in steps]

fig = matplotlib.figure.Figure(figsize=(5, 3.09))
ax = fig.add_subplot()
ax.plot(steps, y)
ax.set_xlabel("timestep")
ax.set_ylabel("L")
ax.tick_params("x", labelrotation=30)
fig

In [ ]:
box_resize = hoomd.update.BoxResize(
    trigger=hoomd.trigger.Periodic(10),
    box=inverse_volume_ramp,
)
simulation.operations.updaters.append(box_resize)

In [ ]:
simulation.run(20001)

In [ ]:
simulation.state.N_particles / simulation.state.box.volume

In [ ]:
render(simulation.state.get_snapshot())

In [ ]:
simulation.operations.updaters.remove(box_resize)

In [ ]:
simulation.run(5e4)

In [ ]:
render(simulation.state.get_snapshot())